In [1]:
import sqlite3
import json
import numpy as np

In [2]:
config_path = '/home/thomasb/albatros_analysis/scripts/orbcomm/config3_corr2.json'
json_path = '/scratch/thomasb/pulsedata_1753200150_len_86260_1760835173.json'
sql_path = '/scratch/thomasb/satellite_data_test.db'

In [3]:
# Connect to the SQLite database (creates file if it doesn't exist)
conn = sqlite3.connect(sql_path)
cur = conn.cursor()

# Create batches table

cur.execute("""
CREATE TABLE IF NOT EXISTS batches (
    batch_start_ts INTEGER PRIMARY KEY,  
    batch_end_ts INTEGER,              
    ingested_at TEXT DEFAULT CURRENT_TIMESTAMP
);
""")

#Create baselines table
cur.execute("""
CREATE TABLE IF NOT EXISTS baselines (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    batch_start_ts INTEGER NOT NULL,
    ant_id INTEGER NOT NULL,  
    ant_name TEXT NOT NULL,        
    consensus_offset INTEGER,
    FOREIGN KEY (batch_start_ts) REFERENCES batches(batch_start_ts) ON DELETE CASCADE,
    UNIQUE (batch_start_ts, ant_id)
);
""")

# Create pulses table
cur.execute("""
CREATE TABLE IF NOT EXISTS pulses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    baseline_id INTEGER NOT NULL,
    start INTEGER NOT NULL,
    end INTEGER NOT NULL,
    sat INTEGER NOT NULL,
    individual_offset INTEGER,
    diff_to_consensus INTEGER,
    FOREIGN KEY (baseline_id) REFERENCES baselines(id) ON DELETE CASCADE,
    UNIQUE (baseline_id, start)
);
""")

# Create satellites table
cur.execute("""
CREATE TABLE IF NOT EXISTS channels (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pulse_id INTEGER NOT NULL,
    chan INTEGER NOT NULL,
    snr INTEGER NOT NULL,
    rel_ratio INTEGER NOT NULL,
    FOREIGN KEY (pulse_id) REFERENCES pulses(id) ON DELETE CASCADE,
    UNIQUE(pulse_id, chan)
);
""")

# Commit changes and close connection
conn.commit()
conn.close()

print("Database created successfully with tables: batches, baselines, pulses, satellites.")


Database created successfully with tables: batches, baselines, pulses, satellites.


In [4]:
ant_names = ['Antenna 1', 'Antenna 2', 'Antenna 3', 'Antenna 4', 'Antenna 5', 'Antenna 6', 'Antenna 7', 'Antenna 8',]

with open(config_path, "r") as f:
    config = json.load(f)
    global_start_time = config["correlation"]["start_timestamp"]
    global_end_time = config["correlation"]["end_timestamp"]

chanlist = np.arange(1834, 1852, dtype=int)
print(global_start_time)

1753200150


In [5]:
conn = sqlite3.connect(sql_path)
cur = conn.cursor()
cur.execute("""
    INSERT OR IGNORE INTO batches (batch_start_ts, batch_end_ts)
    VALUES (?, ?)
""", (global_start_time, global_end_time))


with open(json_path, "r") as f:
    pulsedata_all = json.load(f)
    pulsedata = pulsedata_all[f'{global_start_time}']
    for ant_name, ant_data in pulsedata.items():
        con_off = ant_data['consensus_offset']
        ant_id = ant_names.index(ant_name)
        
        #write baseline into table
        #don't need batch id since it's the primary index, by global_start_time
        cur.execute("""INSERT INTO baselines (batch_start_ts, ant_id, ant_name, consensus_offset)
                        VALUES (?, ?, ?, ?)
                        ON CONFLICT(batch_start_ts, ant_id) DO UPDATE SET
                        consensus_offset = excluded.consensus_offset""", 
                        (global_start_time, ant_id, ant_name, con_off))
        
        #get baseline_id to connect pulses to
        cur.execute("""SELECT id FROM baselines
                        WHERE batch_start_ts = ? AND ant_id = ?""", 
                        (global_start_time, ant_id))
        baseline_id = cur.fetchone()[0]
        
        for pulse in ant_data['pulse_data']:
            start = pulse['start']
            end = pulse['end']
            diff_to_consensus = pulse['diff_to_consensus']
            individual_offset = pulse['individual_offset']
            sat = int(list(pulse['sats_present'].keys())[0])

            #write pulse into table
            cur.execute("""INSERT INTO pulses (baseline_id, start, end, sat, individual_offset, diff_to_consensus)
                            VALUES (?, ?, ?, ?, ?, ?)
                            ON CONFLICT(baseline_id, start) DO UPDATE SET
                            end = excluded.end,
                            sat = excluded.sat,
                            individual_offset = excluded.individual_offset,
                            diff_to_consensus = excluded.diff_to_consensus""", 
                            (baseline_id, start, end, sat, individual_offset, diff_to_consensus))
            
            #get pulse id for writing channel data
            cur.execute("""SELECT id FROM pulses
                            WHERE baseline_id = ? AND start = ?""", 
                            (baseline_id, start))
            pulse_id = cur.fetchone()[0]

            for channel in list(pulse['sats_present'].values())[0]:
                chan = channel[0]
                snr = channel[2]
                rel_ratio = channel[3]

                #write channel into table
                cur.execute("""INSERT INTO channels (pulse_id, chan, snr, rel_ratio)
                                VALUES (?, ?, ?, ?)
                                ON CONFLICT(pulse_id, chan) DO UPDATE SET
                                snr = excluded.snr,
                                rel_ratio = excluded.rel_ratio""", 
                                (pulse_id, chan, snr, rel_ratio))

conn.commit()            
conn.close()

In [6]:
#test entry count in file

conn = sqlite3.connect(sql_path)
cur = conn.cursor()
for table in ["batches", "baselines", "pulses", "channels"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(table, cur.fetchone()[0])
conn.close()

batches 1
baselines 6
pulses 246
channels 390


In [7]:
#test values present

conn = sqlite3.connect(sql_path)
cur = conn.cursor()
cur.execute(f"""
            SELECT * 
            FROM channels
            WHERE snr > 300
            AND rel_ratio > 95
            """)
rows = cur.fetchall()
for row in rows:
    print(row)

conn.close()

(7, 4, 1836, 326, 97)
(8, 4, 1837, 508, 98)
(17, 9, 1836, 416, 98)
(18, 9, 1837, 632, 98)
(20, 10, 1837, 484, 98)
(26, 15, 1836, 344, 97)
(27, 15, 1837, 537, 98)
(29, 16, 1837, 353, 98)
(36, 20, 1837, 358, 97)
(49, 30, 1837, 323, 97)
(57, 35, 1837, 333, 97)
(85, 53, 1837, 368, 97)
(92, 57, 1837, 341, 97)
(128, 81, 1837, 391, 98)
(139, 87, 1836, 432, 98)
(140, 87, 1837, 659, 98)
(146, 92, 1836, 348, 97)
(147, 92, 1837, 552, 98)
(161, 100, 1837, 307, 97)
(184, 115, 1837, 330, 97)
(188, 118, 1837, 385, 97)
(211, 133, 1837, 335, 97)
(264, 168, 1837, 348, 97)
(274, 173, 1836, 409, 98)
(275, 173, 1837, 596, 98)
(277, 174, 1837, 473, 98)
(280, 177, 1836, 327, 97)
(281, 177, 1837, 473, 98)
(294, 185, 1837, 303, 97)
(323, 204, 1837, 326, 97)
(334, 211, 1837, 473, 98)
(342, 216, 1836, 445, 98)
(343, 216, 1837, 696, 98)
(345, 217, 1837, 337, 97)
(349, 220, 1836, 322, 97)
(350, 220, 1837, 499, 98)
(369, 233, 1837, 306, 97)
(377, 237, 1837, 308, 97)
(379, 238, 1837, 337, 97)


In [ ]:
def get_unanimous_detections(sql_path, batch_start_time):
    conn = sqlite3.connect(sql_path)
    cur = conn.cursor()
    cur.execute("""SELECT p.start, p.end
                    FROM pulses p
                    JOIN baselines b ON p.baseline_id = b.id
                    WHERE b.batch_start_ts = ?
                    GROUP BY p.start, p.end
                    HAVING COUNT(DISTINCT b.ant_id) = (
                        SELECT COUNT(DISTINCT ant_id)
                        FROM baselines
                        WHERE batch_start_ts = ?
                    )
                        ORDER BY p.start""", 
                        (batch_start_time, batch_start_time))
    sim_pulses = cur.fetchall()
    return sim_pulses

In [9]:
print(get_unanimous_detections(sql_path, global_start_time))

[(6065, 6510), (6510, 6600), (10545, 10795), (10795, 11085), (16545, 16810), (18030, 18565), (19735, 20295), (21715, 22095), (22830, 23075), (27970, 28075), (28545, 28840), (36180, 36580), (44050, 44510), (52495, 52760), (58510, 58950), (64540, 65055), (72880, 73290), (75120, 75245), (78890, 79395)]


In [10]:
def inspect_pulse(sql_path, batch_start_time, ant_id, pulse_start):
    conn = sqlite3.connect(sql_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT p.id
        FROM pulses p
        JOIN baselines b ON p.baseline_id = b.id
        WHERE b.batch_start_ts = ?
        AND b.ant_id = ?
        AND p.start = ?
    """, (batch_start_time, ant_id, pulse_start))

    result = cur.fetchone()
    if result is None:
        conn.close()
        return []
    
    pulse_id = result[0]

    cur.execute("""SELECT chan, snr, rel_ratio
                FROM channels
                WHERE pulse_id = ?
                ORDER BY chan""", 
                (pulse_id,))
    
    channels = cur.fetchall()

    cur.execute("""SELECT sat
                FROM pulses
                WHERE id = ?
                """,(pulse_id,))
    
    sat = int(cur.fetchall()[0][0])

    conn.close()
    return sat, channels

In [12]:
sat, channels = inspect_pulse(sql_path, global_start_time, 3, 6510)
print(sat)
print(channels)

25338
[(1841, 310, 4), (1842, 33, 9), (1846, 185, 43)]
